In [1]:
from pathlib import Path
from PIL import Image
import os

def check_images(design_ids):
    oversized_dimensions = []
    oversized_files = []
    
    for design_id in design_ids:
        design_path = Path(f"scraped_designs/{design_id}")
        if not design_path.exists():
            print(f"Design folder {design_id} not found")
            continue
            
        # Check all PNG files in the design folder
        for png_file in design_path.glob("*.png"):
            try:
                # Check file size
                file_size_mb = os.path.getsize(png_file) / (1024 * 1024)
                if file_size_mb > 5:
                    oversized_files.append((design_id, png_file.name, f"{file_size_mb:.2f}MB"))
                
                # Check dimensions
                with Image.open(png_file) as img:
                    width, height = img.size
                    if width > 8000 or height > 8000:
                        oversized_dimensions.append((design_id, png_file.name, f"{width}x{height}"))
                        
            except Exception as e:
                print(f"Error processing {png_file}: {str(e)}")
    
    return oversized_dimensions, oversized_files

# Check the specified designs
design_ids = ["015", "016", "042", "099", "181"]
dimensions, files = check_images(design_ids)

print("\nImages with dimensions over 8000 pixels:")
for design_id, filename, dimensions in dimensions:
    print(f"Design {design_id}: {filename} ({dimensions})")

print("\nImages over 5MB:")
for design_id, filename, size in files:
    print(f"Design {design_id}: {filename} ({size})")


Images with dimensions over 8000 pixels:
Design 015: screenshot_mobile.png (712x16242)
Design 016: screenshot_mobile.png (480x12367)
Design 042: screenshot_mobile.png (502x10447)
Design 181: screenshot_mobile.png (667x15794)

Images over 5MB:


## Image Size Check and Crop

Let's check the specified designs for oversized images and crop them if necessary.

In [1]:
from pathlib import Path
from PIL import Image
import os

def check_and_crop_images(design_ids, max_dimension=8000):
    oversized_dimensions = []
    oversized_files = []
    cropped_files = []
    
    for design_id in design_ids:
        design_path = Path(f"scraped_designs/{design_id}")
        if not design_path.exists():
            print(f"Design folder {design_id} not found")
            continue
            
        # Check all PNG files in the design folder
        for png_file in design_path.glob("*.png"):
            try:
                # Check file size
                file_size_mb = os.path.getsize(png_file) / (1024 * 1024)
                if file_size_mb > 5:
                    oversized_files.append((design_id, png_file.name, f"{file_size_mb:.2f}MB"))
                
                # Check dimensions and crop if necessary
                with Image.open(png_file) as img:
                    width, height = img.size
                    if width > max_dimension or height > max_dimension:
                        oversized_dimensions.append((design_id, png_file.name, f"{width}x{height}"))
                        
                        # Calculate new dimensions while preserving aspect ratio
                        if width > height:
                            new_width = max_dimension
                            new_height = int(height * (max_dimension / width))
                        else:
                            new_height = max_dimension
                            new_width = int(width * (max_dimension / height))
                        
                        # Resize the image
                        resized_img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
                        
                        # Save the resized image
                        resized_path = png_file.parent / f"{png_file.stem}{png_file.suffix}"
                        resized_img.save(resized_path, optimize=True, quality=95)
                        cropped_files.append((design_id, png_file.name, f"{new_width}x{new_height}"))
                        
            except Exception as e:
                print(f"Error processing {png_file}: {str(e)}")
    
    return oversized_dimensions, oversized_files, cropped_files

# Check and crop the specified designs
design_ids = ["015", "016", "042", "099", "181"]
dimensions, files, cropped = check_and_crop_images(design_ids)

print("\nImages with dimensions over 8000 pixels:")
for design_id, filename, dimensions in dimensions:
    print(f"Design {design_id}: {filename} ({dimensions})")

print("\nImages over 5MB:")
for design_id, filename, size in files:
    print(f"Design {design_id}: {filename} ({size})")

print("\nCropped images:")
for design_id, filename, new_dimensions in cropped:
    print(f"Design {design_id}: {filename} -> {filename.replace('.png', '_resized.png')} ({new_dimensions})")


Images with dimensions over 8000 pixels:
Design 015: screenshot_mobile.png (712x16242)
Design 016: screenshot_mobile.png (480x12367)
Design 042: screenshot_mobile.png (502x10447)
Design 181: screenshot_mobile.png (667x15794)

Images over 5MB:

Cropped images:
Design 015: screenshot_mobile.png -> screenshot_mobile_resized.png (350x8000)
Design 016: screenshot_mobile.png -> screenshot_mobile_resized.png (310x8000)
Design 042: screenshot_mobile.png -> screenshot_mobile_resized.png (384x8000)
Design 181: screenshot_mobile.png -> screenshot_mobile_resized.png (337x8000)
